# LAB 13 - 웹 페이지 데이터 수집하기 연구과제
https://data.hossam.kr/py/myfood.html 웹페이지데이터수집하기
- 주어진 페이지에 메뉴명,메뉴설명, 메뉴이미지를 수집하여 엑셀형식으로 저장하세요.

- 메뉴 이미지는 별도의 폴더에 다운로드 받도록 구현하세요.

<작업 순서>
- STEP 1. 라이브러리 불러오기
- STEP 2. 데이터 요청하기
- STEP 3. 데이터 확인하기
- STEP 4. 불러온 데이터에 대해 객체 생성하기
- STEP 5. 메뉴명, 메뉴 설명, 메뉴 이미지 데이터 가져오기
- STEP 6. 데이터 프레임 생성을 위해 데이터 정제하기
- STEP 7. 데이터 프레임 생성하기
- STEP 8. 이미지 다운을 위한 파일 다운로드 함수 정의하기
- STEP 9. 비동기식으로 다운로드 처리하기

### 1. 라이브러리 불러오기

In [79]:
import requests
from bs4 import BeautifulSoup
from pandas import DataFrame

### 2. 데이터 요청하기

In [ ]:
#웹에 데이터 요청하기
with requests.Session() as session:

  #세션 객체에 웹 브라우저 정보 (UserAgent) 주입 (웹서버가 파이썬 프로그램을 정상적인 웹 브라우저로 여기도록)
  session.headers.update({"User-Agent":"Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/51.0.2704.103 Safari/537.36"
                          })
  


  url = "https://data.hossam.kr/py/myfood.html"
  r=session.get(url,stream=True)    #이미지 파일 다운로드를 위해 stream=True 설정
  print(r.url)
  

  if r.status_code !=200:
    msg ="[%d Error] %s 에러가 발생함" % (r.status_code,r.reason)
    raise Exception(msg)

print(r) # HTTP 통신 상태 확인


https://data.hossam.kr/py/myfood.html
<Response [200]>


### 3.데이터 확인하기

In [7]:
r.encoding ='utf-8' 
print(type(r.text))
r.text

<class 'str'>


'<!DOCTYPE html><html lang="ko" translate="no"><head><meta charset="UTF-8"><meta name="google" content="notranslate" /><meta name="viewport" content="width=device-width, initial-scale=1.0"><title>FoodBlog</title><link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Noto+Sans+KR:wght@100;300;400;500;700;900&display=swap" /><link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/6.6.0/css/all.min.css" /><link rel="stylesheet" href="https://preview.hossam.kr/html/myfood/assets/css/reset.css" /><link rel="stylesheet" href="https://preview.hossam.kr/html/myfood/assets/css/common.css" /><link rel="stylesheet" href="https://preview.hossam.kr/html/myfood/assets/css/index.css" /></head><body><div class="container"><header class="header"><div class="content-container"><a href="#" class="icon-button left"><i class="fa-solid fa-bars"></i></a><h1 class="logo">My Food</h1><a href="#" class="icon-button right"><i class="fa-solid fa-envelope"></i></a></div></he

### 4. 불러온 데이터 객체 생성하기

In [9]:
soup =BeautifulSoup(r.text)
print(type(soup))
soup

<class 'bs4.BeautifulSoup'>


<!DOCTYPE html>
<html lang="ko" translate="no"><head><meta charset="utf-8"/><meta content="notranslate" name="google"><meta content="width=device-width, initial-scale=1.0" name="viewport"/><title>FoodBlog</title><link href="https://fonts.googleapis.com/css2?family=Noto+Sans+KR:wght@100;300;400;500;700;900&amp;display=swap" rel="stylesheet"/><link href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/6.6.0/css/all.min.css" rel="stylesheet"/><link href="https://preview.hossam.kr/html/myfood/assets/css/reset.css" rel="stylesheet"/><link href="https://preview.hossam.kr/html/myfood/assets/css/common.css" rel="stylesheet"/><link href="https://preview.hossam.kr/html/myfood/assets/css/index.css" rel="stylesheet"/></meta></head><body><div class="container"><header class="header"><div class="content-container"><a class="icon-button left" href="#"><i class="fa-solid fa-bars"></i></a><h1 class="logo">My Food</h1><a class="icon-button right" href="#"><i class="fa-solid fa-envelope"></i></a></di

### 5-1 타이틀 데이터 가져오기

In [ ]:
food_title =soup.select("h2")
food_title

[<h2>The Perfect Sandwich, A Real NYC Classic</h2>,
 <h2>Let Me Tell You About This Steak</h2>,
 <h2>Cherries, interrupted</h2>,
 <h2>Once Again, Robust Wine and Vegetable Pasta</h2>,
 <h2>All I Need Is a Popsicle</h2>,
 <h2>Salmon For Your Skin</h2>,
 <h2>The Perfect Sandwich, A Real Classic</h2>,
 <h2>Le French</h2>,
 <h2>About Me, The Food Man</h2>]

In [18]:
titles=[]
for title in food_title:
  a=title.text.strip()
  titles.append(a)

titles

['The Perfect Sandwich, A Real NYC Classic',
 'Let Me Tell You About This Steak',
 'Cherries, interrupted',
 'Once Again, Robust Wine and Vegetable Pasta',
 'All I Need Is a Popsicle',
 'Salmon For Your Skin',
 'The Perfect Sandwich, A Real Classic',
 'Le French',
 'About Me, The Food Man']

### 5-2 디스크립션 데이터 가져오기

In [24]:
food_desc =soup.select("div.food-content p")
food_desc

[<p>Just some random text, lorem ipsum text praesent tincidunt ipsum lipsum</p>,
 <p>Once again, some random text to lorem lorem lorem lorem ipsum text praesent tincidunt ipsum lipsum.</p>,
 <p>Lorem ipsum text praesent tincidunt ipsum lipsum.</p>,
 <p>Lorem ipsum text praesent tincidunt ipsum lipsum.</p>,
 <p>Lorem ipsum text praesent tincidunt ipsum lipsum.</p>,
 <p>Once again, some random text to lorem lorem lorem lorem ipsum text praesent tincidunt ipsum lipsum.</p>,
 <p>Just some random text, lorem ipsum text praesent tincidunt ipsum lipsum.</p>,
 <p>Lorem lorem lorem lorem ipsum text praesent tincidunt ipsum lipsum.</p>]

In [25]:
descriptions=[]
for description in food_desc:
  a=description.text.strip()
  descriptions.append(a)

descriptions

['Just some random text, lorem ipsum text praesent tincidunt ipsum lipsum',
 'Once again, some random text to lorem lorem lorem lorem ipsum text praesent tincidunt ipsum lipsum.',
 'Lorem ipsum text praesent tincidunt ipsum lipsum.',
 'Lorem ipsum text praesent tincidunt ipsum lipsum.',
 'Lorem ipsum text praesent tincidunt ipsum lipsum.',
 'Once again, some random text to lorem lorem lorem lorem ipsum text praesent tincidunt ipsum lipsum.',
 'Just some random text, lorem ipsum text praesent tincidunt ipsum lipsum.',
 'Lorem lorem lorem lorem ipsum text praesent tincidunt ipsum lipsum.']

### 5-3. 이미지 데이터 가져오기

In [61]:
food_img =soup.select("div.img-wrapper img")
food_img


[<img src="https://preview.hossam.kr/html/myfood/assets/img/sandwich.jpg"/>,
 <img src="https://preview.hossam.kr/html/myfood/assets/img/steak.jpg"/>,
 <img src="https://preview.hossam.kr/html/myfood/assets/img/cherries.jpg"/>,
 <img src="https://preview.hossam.kr/html/myfood/assets/img/wine.jpg"/>,
 <img src="https://preview.hossam.kr/html/myfood/assets/img/popsicle.jpg"/>,
 <img src="https://preview.hossam.kr/html/myfood/assets/img/salmon.jpg"/>,
 <img src="https://preview.hossam.kr/html/myfood/assets/img/sandwich.jpg"/>,
 <img src="https://preview.hossam.kr/html/myfood/assets/img/croissant.jpg"/>]

In [ ]:
images=[]
for image in food_img:
  a=image.get("src")
  # print(image.get("src"))

  images.append(a)

print(images)






['https://preview.hossam.kr/html/myfood/assets/img/sandwich.jpg', 'https://preview.hossam.kr/html/myfood/assets/img/steak.jpg', 'https://preview.hossam.kr/html/myfood/assets/img/cherries.jpg', 'https://preview.hossam.kr/html/myfood/assets/img/wine.jpg', 'https://preview.hossam.kr/html/myfood/assets/img/popsicle.jpg', 'https://preview.hossam.kr/html/myfood/assets/img/salmon.jpg', 'https://preview.hossam.kr/html/myfood/assets/img/sandwich.jpg', 'https://preview.hossam.kr/html/myfood/assets/img/croissant.jpg']


### 6. titles , descriptions, images 데이터 프레임 형식으로 만들기 위한 정제하기
- 데이터 프레임으로 만들기 위해서는 딕셔너리를 원소로 가지는 리스트를 생성해야한다
- 

In [ ]:

overview=[]
for i in range(0,len(titles)-1):
  overall = {"title":titles[i],"descriptions":descriptions[i],"images":images[i]}
  print(overall)
  overview.append(overall)

print(overview)

 



{'title': 'The Perfect Sandwich, A Real NYC Classic', 'descriptions': 'Just some random text, lorem ipsum text praesent tincidunt ipsum lipsum', 'images': 'https://preview.hossam.kr/html/myfood/assets/img/sandwich.jpg'}
{'title': 'Let Me Tell You About This Steak', 'descriptions': 'Once again, some random text to lorem lorem lorem lorem ipsum text praesent tincidunt ipsum lipsum.', 'images': 'https://preview.hossam.kr/html/myfood/assets/img/steak.jpg'}
{'title': 'Cherries, interrupted', 'descriptions': 'Lorem ipsum text praesent tincidunt ipsum lipsum.', 'images': 'https://preview.hossam.kr/html/myfood/assets/img/cherries.jpg'}
{'title': 'Once Again, Robust Wine and Vegetable Pasta', 'descriptions': 'Lorem ipsum text praesent tincidunt ipsum lipsum.', 'images': 'https://preview.hossam.kr/html/myfood/assets/img/wine.jpg'}
{'title': 'All I Need Is a Popsicle', 'descriptions': 'Lorem ipsum text praesent tincidunt ipsum lipsum.', 'images': 'https://preview.hossam.kr/html/myfood/assets/img/

### 7. 데이터 프레임으로 만들기

In [82]:
df = DataFrame(overview)
df.to_excel("html 연구과제.xlsx")
df



,title,descriptions,images
0,"The Perfect Sandwich, A Real NYC Classic","Just some random text, lorem ipsum text praese...",https://preview.hossam.kr/html/myfood/assets/i...
1,Let Me Tell You About This Steak,"Once again, some random text to lorem lorem lo...",https://preview.hossam.kr/html/myfood/assets/i...
2,"Cherries, interrupted",Lorem ipsum text praesent tincidunt ipsum lipsum.,https://preview.hossam.kr/html/myfood/assets/i...
3,"Once Again, Robust Wine and Vegetable Pasta",Lorem ipsum text praesent tincidunt ipsum lipsum.,https://preview.hossam.kr/html/myfood/assets/i...
4,All I Need Is a Popsicle,Lorem ipsum text praesent tincidunt ipsum lipsum.,https://preview.hossam.kr/html/myfood/assets/i...
5,Salmon For Your Skin,"Once again, some random text to lorem lorem lo...",https://preview.hossam.kr/html/myfood/assets/i...
6,"The Perfect Sandwich, A Real Classic","Just some random text, lorem ipsum text praese...",https://preview.hossam.kr/html/myfood/assets/i...
7,Le French,Lorem lorem lorem lorem ipsum text praesent ti...,https://preview.hossam.kr/html/myfood/assets/i...


### 8. 이미지 다운로드 받기 위해, 파일 다운로드 함수 정의
- 여기서 download 함수는 url 에 있는 파일을 get 해와서, target 으로 항는 파일 디렉토리에 저장하는 역할을 수행


In [ ]:
def download(url,target):

  session.headers.update({"User-Agent":"Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/51.0.2704.103 Safari/537.36"
                          })
  
  try:
    #이미지를 웹 서버에 요청
    r=session.get(url,stream=True)
    r.encoding = 'utf-8'

    #서버가 보낸 그림 데이터를 가져와서 저장. 글자가 아닌 이미지 데이터기 떄문에 wb 모드로 저장
    with open(target,"wb") as f:
      f.write(r.raw.read())
      print(target, "가 저장되었습니다")

  #예외처리
  except Exception:
    print(target,"저장 실패", e)



### 9. 비동기식 다운로드 처리
- 현재 시각으로 된 새 폴더를 만든다.
- ThreadPoolExecutor로 스레드 풀을 열어 동시에 여러 다운로드 작업을 실행한다.
- overview에 담긴 각 아이템에서 이미지 URL을 꺼내 파일 경로를 만들고, download(url, path)를 비동기 제출한다.

In [ ]:
import os
import datetime as dt

#비동기 처리 기능을 제공하는 모듈
from concurrent import futures

#다운로드 결과가 저장될 폴더 생성하기 (폴더 이름은 현재 시간으로 생성)
dirname = dt.datetime.now().strftime("%Y%m%d-%H%M%S")
os.mkdir(dirname)


#비동기 작업 내에서 이미지 다운로드 받기 - 비동기 병렬 작업을 시작 (여러개의 스레드를 만들어 여러 작업을 동시에 실행하는 역할)
with futures.ThreadPoolExecutor() as executor:


  #이미지 정보를 하나씩 꺼내서 해당 이미지에 대한 파일 이름을 만들고, 해당 파일 경로를 기준으로 파일을 저장하는 download 함수를 실행 (함수와 그 함수에 필요한 파라미터를 전송하여 비동기 처리 되도록 함)
  for i,v in enumerate(overview):

    #저장될 파일 경로 문자열 생성
    file_path = os.path.join(dirname, "%d.jpg" %i)

    # 비동기 다운로드 요청
    executor.submit(download,v['images'],file_path)

20251107-165739\1.jpg 가 저장되었습니다
20251107-165739\4.jpg 가 저장되었습니다
20251107-165739\5.jpg 가 저장되었습니다
20251107-165739\6.jpg 가 저장되었습니다
20251107-165739\3.jpg 가 저장되었습니다
20251107-165739\0.jpg 가 저장되었습니다
20251107-165739\7.jpg 가 저장되었습니다
20251107-165739\2.jpg 가 저장되었습니다
